# Cuarta Entrega - Visualización e Integración Interactiva
## Gramática de Gráficos con Altair para Análisis de Tenis

**Proyecto:** Predicción de Duración de Partidos de Tenis  
**Fecha:** 5 de Noviembre de 2025  
**Alumno:** Juan Ignacio Barranco Bastan  
**Objetivo:** Visualizaciones interactivas y comunicación de hallazgos

---

## 📊 VISUALIZACIONES INCLUIDAS

✅ **Visualización 1:** Scatter Plot Interactivo - Predicciones vs Realidad  
✅ **Visualización 2:** Heatmap Dinámico - Performance por Segmento  
✅ **Visualización 3:** Dashboard Multi-Panel - Comparativa Modelos  
✅ **Principios aplicados:** Gramática de gráficos, comparabilidad, expresividad

---

In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
import altair as alt
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

# Configuración de Altair
alt.data_transformers.enable('json')
alt.themes.enable('opaque')

print("📊 Notebook de Visualización Interactiva - Cuarta Entrega")
print(f"Altair version: {alt.__version__}")
print(f"Pandas version: {pd.__version__}")

## 1. Preparación de Datos y Modelos

Vamos a cargar los datos y entrenar rápidamente los mejores modelos identificados en la entrega anterior.

In [ ]:
# Cargar dataset
df = pd.read_csv('/kaggle/input/matches-cleaned-cdd-proyecto/matches_cleaned.csv')

# Features originales
features_originales = [
    'tourney_level', 'surface', 'round', 'best_of',
    'winner_rank', 'loser_rank',
    'winner_age', 'loser_age',
    'winner_hand', 'loser_hand',
    'winner_ht', 'loser_ht'
]

target = 'minutes'

# Preparar dataset
df_model = df[features_originales + [target]].copy()
df_model = df_model.dropna(subset=[target])
df_model = df_model[df_model[target] > 0]

print(f"✅ Dataset: {df_model.shape[0]} partidos, {df_model.shape[1]} variables")
print(f"Duración promedio: {df_model[target].mean():.1f} minutos")
print(f"Rango: {df_model[target].min():.0f} - {df_model[target].max():.0f} minutos")

In [ ]:
# Ingeniería de Features (basada en la entrega anterior)
df_eng = df_model.copy()

# Crear features derivadas
df_eng['rank_diff'] = np.abs(df_eng['winner_rank'] - df_eng['loser_rank'])
df_eng['rank_avg'] = (df_eng['winner_rank'] + df_eng['loser_rank']) / 2
df_eng['age_diff'] = np.abs(df_eng['winner_age'] - df_eng['loser_age'])
df_eng['ht_diff'] = np.abs(df_eng['winner_ht'] - df_eng['loser_ht'])
df_eng['is_grand_slam'] = df_eng['tourney_level'].isin(['G']).astype(int)
df_eng['same_hand'] = (df_eng['winner_hand'] == df_eng['loser_hand']).astype(int)
df_eng['fast_surface'] = df_eng['surface'].isin(['Grass', 'Hard']).astype(int)

# Crear categorías de duración para clasificación
df_eng['duration_category'] = pd.cut(df_eng[target], 
                                   bins=[0, 100, 150, float('inf')], 
                                   labels=['CORTO', 'MEDIO', 'LARGO'])

# Features finales
features_finales = features_originales + [
    'rank_diff', 'rank_avg', 'age_diff', 'ht_diff',
    'is_grand_slam', 'same_hand', 'fast_surface'
]

print(f"✅ Features creadas: {len(features_finales)} variables")
print(f"✅ Categorías de duración: {df_eng['duration_category'].value_counts().to_dict()}")

In [ ]:
# División train/test
X = df_eng[features_finales]
y_reg = df_eng[target]  # Para regresión
y_class = df_eng['duration_category']  # Para clasificación

X_train, X_test, y_reg_train, y_reg_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42, stratify=df_eng['best_of']
)

_, _, y_class_train, y_class_test = train_test_split(
    X, y_class, test_size=0.2, random_state=42, stratify=df_eng['best_of']
)

print(f"✅ División realizada: {X_train.shape[0]} train, {X_test.shape[0]} test")

In [ ]:
# Crear preprocessor
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Entrenar los mejores modelos (basado en análisis anterior)
print("🚀 Entrenando mejores modelos...")

# Modelo de regresión: GradientBoosting (mejor según análisis previo)
model_regresion = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(n_estimators=100, random_state=42))
])

model_regresion.fit(X_train, y_reg_train)
y_reg_pred = model_regresion.predict(X_test)

# Modelo de clasificación: GradientBoostingClassifier
model_clasificacion = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GradientBoostingClassifier(n_estimators=100, random_state=42))
])

model_clasificacion.fit(X_train, y_class_train)
y_class_pred = model_clasificacion.predict(X_test)

# Métricas
rmse = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred))
r2 = r2_score(y_reg_test, y_reg_pred)
accuracy = accuracy_score(y_class_test, y_class_pred)

print(f"✅ Regresión - RMSE: {rmse:.2f} min, R²: {r2:.4f}")
print(f"✅ Clasificación - Accuracy: {accuracy:.4f}")

## 2. VISUALIZACIÓN 1: Scatter Plot Interactivo - Predicciones vs Realidad

**Principios aplicados:**
- **Expresividad:** Cada punto representa una predicción vs valor real
- **Comparabilidad:** Línea diagonal de referencia para evaluar precisión
- **Interactividad:** Zoom, selección y tooltips informativos

In [ ]:
# Preparar datos para visualización
df_viz1 = pd.DataFrame({
    'Duración Real': y_reg_test,
    'Duración Predicha': y_reg_pred,
    'Error Absoluto': np.abs(y_reg_test - y_reg_pred),
    'Superficie': X_test['surface'],
    'Nivel Torneo': X_test['tourney_level'],
    'Grand Slam': X_test['is_grand_slam'].map({0: 'No', 1: 'Sí'}),
    'Mejor de': X_test['best_of']
})

# Crear selector de superficie (CORREGIDO PARA ALTAIR 5.x)
# Cambio: selection_multi puede seguir funcionando, pero verificamos
surface_selector = alt.selection_point(fields=['Superficie'], toggle=True)

# Gráfico principal
scatter = alt.Chart(df_viz1).mark_circle(size=60, opacity=0.7).add_selection(
    surface_selector
).encode(
    x=alt.X('Duración Real:Q', 
            title='Duración Real (minutos)',
            scale=alt.Scale(domain=[50, 400])),
    y=alt.Y('Duración Predicha:Q', 
            title='Duración Predicha (minutos)',
            scale=alt.Scale(domain=[50, 400])),
    color=alt.Color('Superficie:N', 
                   title='Superficie',
                   scale=alt.Scale(range=['#1f77b4', '#ff7f0e', '#2ca02c'])),
    size=alt.Size('Error Absoluto:Q', 
                 title='Error (min)',
                 scale=alt.Scale(range=[30, 200])),
    stroke=alt.value('white'),
    strokeWidth=alt.value(0.5),
    opacity=alt.condition(surface_selector, alt.value(0.8), alt.value(0.3)),
    tooltip=[
        alt.Tooltip('Duración Real:Q', title='Real (min)', format='.1f'),
        alt.Tooltip('Duración Predicha:Q', title='Predicha (min)', format='.1f'),
        alt.Tooltip('Error Absoluto:Q', title='Error (min)', format='.1f'),
        alt.Tooltip('Superficie:N', title='Superficie'),
        alt.Tooltip('Grand Slam:N', title='Grand Slam'),
        alt.Tooltip('Mejor de:O', title='Mejor de')
    ]
).properties(
    title=alt.TitleParams(
        text=["Predicciones vs Realidad - Modelo de Regresión",
              "Tamaño = Error | Color = Superficie | Click para filtrar"],
        fontSize=16,
        anchor='start'
    ),
    width=500,
    height=400
)

# Línea de referencia (predicción perfecta)
line = alt.Chart(pd.DataFrame({
    'x': [50, 400],
    'y': [50, 400]
})).mark_line(
    color='red',
    strokeDash=[5, 5],
    size=2
).encode(
    x='x:Q',
    y='y:Q'
)

# Leyenda interactiva
legend = alt.Chart(df_viz1).mark_rect().add_selection(
    surface_selector
).encode(
    y=alt.Y('Superficie:N', axis=alt.Axis(orient='right', title='Filtrar por Superficie')),
    color=alt.condition(surface_selector, 
                       alt.Color('Superficie:N', legend=None), 
                       alt.value('lightgray'))
).properties(
    width=100,
    height=100,
    title='Click para filtrar'
)

# Combinar gráficos
viz1 = alt.hconcat(
    (scatter + line).resolve_scale(color='independent'),
    legend
).resolve_legend(
    color="independent"
)

print("📊 VISUALIZACIÓN 1: Scatter Plot Interactivo creado")
viz1

## 3. VISUALIZACIÓN 2: Heatmap Dinámico - Performance por Segmento

**Principios aplicados:**
- **Comparabilidad:** Matriz que permite comparar performance entre categorías
- **Expresividad:** Color intenso = mayor error, facilita identificación de problemas
- **Adaptabilidad:** Diferentes métricas según el tipo de análisis

**Notas sobre la visualización:**
- 📌 **Celdas sin color (vacías):** Representan combinaciones Superficie × Nivel con menos de 5 partidos en la muestra. No se muestran para evitar promedios poco confiables.
- 📌 **Bar chart inferior:** Muestra la cantidad de partidos del test set por superficie (puede diferir del total del dataset).

In [ ]:
# Calcular errores por segmento
df_segmentos = pd.DataFrame({
    'Real': y_reg_test,
    'Predicha': y_reg_pred,
    'Error': np.abs(y_reg_test - y_reg_pred),
    'Superficie': X_test['surface'].values,
    'Nivel': X_test['tourney_level'].values,
    'Ronda': X_test['round'].values,
    'Mejor_de': X_test['best_of'].values
})

# Mapear niveles de torneo a nombres más claros
nivel_map = {
    'G': 'Grand Slam',
    'M': 'Masters',
    'A': 'ATP 250/500',
    'C': 'Challenger',
    'F': 'Futures',
    'D': 'Davis Cup'
}
df_segmentos['Nivel_Nombre'] = df_segmentos['Nivel'].map(nivel_map).fillna('Otro')

# Crear matriz de errores promedio
error_matrix = df_segmentos.groupby(['Superficie', 'Nivel_Nombre'])['Error'].agg([
    ('Error_Promedio', 'mean'),
    ('Cantidad_Partidos', 'count')
]).reset_index()

# Filtrar combinaciones con al menos 5 partidos
# NOTA: Celdas sin estas combinaciones aparecerán VACÍAS en el heatmap
MIN_PARTIDOS = 5
error_matrix = error_matrix[error_matrix['Cantidad_Partidos'] >= MIN_PARTIDOS]

print(f"📊 Matriz de errores: {len(error_matrix)} combinaciones Superficie × Nivel")
print(f"   (Filtradas: mínimo {MIN_PARTIDOS} partidos por combinación)")
print(error_matrix.head())

In [ ]:
# Selector para métrica (CORREGIDO - value debe ser un ARRAY)
# El error anterior: value={'metric': 'Error_Promedio'} está mal
# Correcto: value=[{'metric': 'Error_Promedio'}] ← Array con diccionario
metric_selector = alt.selection_point(
    fields=['metric'],
    value=[{'metric': 'Error_Promedio'}]  # ← AHORA ES UN ARRAY
)

# Base chart
base = alt.Chart(error_matrix)

# Heatmap principal
heatmap = base.mark_rect().encode(
    x=alt.X('Superficie:O', title='Superficie', sort=['Clay', 'Hard', 'Grass']),
    y=alt.Y('Nivel_Nombre:O', 
            title='Nivel de Torneo', 
            sort=['Grand Slam', 'Masters', 'ATP 250/500', 'Challenger', 'Futures']),
    color=alt.Color('Error_Promedio:Q',
                   title='Error Promedio (min)',
                   scale=alt.Scale(
                       scheme='reds',
                       domain=[error_matrix['Error_Promedio'].min(), 
                              error_matrix['Error_Promedio'].max()]
                   )),
    stroke=alt.value('white'),
    strokeWidth=alt.value(2),
    tooltip=[
        alt.Tooltip('Superficie:O', title='Superficie'),
        alt.Tooltip('Nivel_Nombre:O', title='Nivel'),
        alt.Tooltip('Error_Promedio:Q', title='Error Promedio (min)', format='.2f'),
        alt.Tooltip('Cantidad_Partidos:Q', title='Cantidad de Partidos')
    ]
).properties(
    title=alt.TitleParams(
        text=["Mapa de Calor: Error de Predicción por Superficie y Nivel",
              "Rojo intenso = Mayor error | Celdas vacías = Datos insuficientes (<5 partidos)"],
        fontSize=14,
        anchor='start'
    ),
    width=400,
    height=300
)

# Texto con valores
text = base.mark_text(
    align='center',
    baseline='middle',
    fontSize=12,
    fontWeight='bold',
    color='white'
).encode(
    x=alt.X('Superficie:O', sort=['Clay', 'Hard', 'Grass']),
    y=alt.Y('Nivel_Nombre:O', 
            sort=['Grand Slam', 'Masters', 'ATP 250/500', 'Challenger', 'Futures']),
    text=alt.Text('Error_Promedio:Q', format='.1f')
)

# Chart de cantidad de partidos (complementario)
count_chart = base.mark_bar().encode(
    x=alt.X('Superficie:O', title=''),
    y=alt.Y('Cantidad_Partidos:Q', title='Partidos en muestra (test set)'),
    color=alt.Color('Superficie:N', legend=None),
    tooltip=[
        alt.Tooltip('Superficie:O'),
        alt.Tooltip('Cantidad_Partidos:Q', title='Total Partidos')
    ]
).properties(
    title="Distribución de partidos por superficie (solo partidos con datos completos)",
    width=400,
    height=150
)

# Combinar visualizaciones
viz2 = alt.vconcat(
    (heatmap + text),
    count_chart
).resolve_scale(
    color='independent'
)

print("📊 VISUALIZACIÓN 2: Heatmap Dinámico creado")
print("   ✅ Notas: Celdas vacías = <5 partidos en esa combinación")
viz2

## 4. VISUALIZACIÓN 3: Dashboard Multi-Panel - Comparativa Modelos

**Principios aplicados:**
- **Gramática de gráficos:** Diferentes encodings para diferentes aspectos de los datos
- **Comparabilidad:** Paneles alineados permiten comparación directa
- **Expresividad:** Cada panel comunica un aspecto específico del rendimiento

In [ ]:
# Preparar datos para comparativa de modelos
# IMPORTANTE: Solo usar datos del modelo que realmente entrenamos (Gradient Boosting)
# Los demás son para referencia histórica de entregas anteriores

# Datos basados en análisis real de entregas anteriores + actual
# Fuente: ANALISIS_COMPLETO_3ERA_ENTREGA.md + entrenamiento actual
modelo_metrics = pd.DataFrame({
    'Modelo': ['Ridge', 'Random Forest', 'Gradient Boosting', 'XGBoost', 'LightGBM'],
    'RMSE_Train': [42.15, 28.94, 31.94, 26.30, 25.58],      # Entrenamientos anteriores
    'RMSE_Test': [41.96, 36.01, 35.84, 39.08, 36.76],       # Resultados en test
    'R2_Train': [0.3125, 0.6896, 0.5715, 0.6987, 0.6253],
    'R2_Test': [0.2697, 0.2436, 0.2613, 0.1817, 0.2329],
    'Tipo': ['Lineal', 'Ensemble', 'Ensemble', 'Boosting', 'Boosting'],
    'Entrenado_Actual': [False, False, True, False, False]   # Solo GB en Kaggle actual
})

# Calcular overfitting (diferencia entre train y test)
modelo_metrics['Overfitting_RMSE'] = modelo_metrics['RMSE_Train'] - modelo_metrics['RMSE_Test']
modelo_metrics['Overfitting_R2'] = modelo_metrics['R2_Train'] - modelo_metrics['R2_Test']

# Marcar el mejor modelo (Gradient Boosting)
best_model_idx = modelo_metrics[modelo_metrics['Modelo'] == 'Gradient Boosting'].index[0]
modelo_metrics['Es_Mejor'] = modelo_metrics.index == best_model_idx

print("📊 Métricas de modelos (comparativa histórica + actual):")
print(modelo_metrics[['Modelo', 'RMSE_Test', 'R2_Test', 'Entrenado_Actual']].sort_values('RMSE_Test'))
print("\n✅ Modelo actualmente en Kaggle: Gradient Boosting")
print(f"   RMSE Test: {modelo_metrics.loc[best_model_idx, 'RMSE_Test']:.2f} minutos")
print(f"   R²: {modelo_metrics.loc[best_model_idx, 'R2_Test']:.4f}")

In [ ]:
# Selector de modelo (CORREGIDO PARA ALTAIR 5.x)
# Cambio: selection_multi → selection_point con toggle=True
model_selector = alt.selection_point(fields=['Modelo'], toggle=True)

# Panel 1: RMSE Comparison (con indicador del modelo actual)
rmse_chart = alt.Chart(modelo_metrics).mark_bar().add_selection(
    model_selector
).encode(
    x=alt.X('RMSE_Test:Q', title='RMSE Test (minutos)', scale=alt.Scale(domain=[0, 45])),
    y=alt.Y('Modelo:O', title='', sort=alt.EncodingSortField(field='RMSE_Test', order='ascending')),
    color=alt.condition(
        model_selector,
        alt.Color('Tipo:N', 
                 title='Tipo de Modelo',
                 scale=alt.Scale(range=['#1f77b4', '#ff7f0e', '#2ca02c'])),
        alt.value('lightgray')
    ),
    stroke=alt.condition(
        alt.datum.Es_Mejor,
        alt.value('gold'),
        alt.value('white')
    ),
    strokeWidth=alt.condition(
        alt.datum.Es_Mejor,
        alt.value(4),
        alt.value(1)
    ),
    opacity=alt.condition(
        alt.datum.Entrenado_Actual,
        alt.value(1),
        alt.value(0.6)
    ),
    tooltip=[
        alt.Tooltip('Modelo:N'),
        alt.Tooltip('RMSE_Test:Q', title='RMSE Test', format='.2f'),
        alt.Tooltip('R2_Test:Q', title='R² Test', format='.4f'),
        alt.Tooltip('Tipo:N'),
        alt.Tooltip('Entrenado_Actual:N', title='Entrenado Kaggle')
    ]
).properties(
    title="Error (RMSE) por Modelo (Borde dorado = mejor)",
    width=300,
    height=200
)

# Panel 2: R² vs RMSE Scatter
scatter_metrics = alt.Chart(modelo_metrics).mark_circle(size=120).add_selection(
    model_selector
).encode(
    x=alt.X('R2_Test:Q', title='R² Test', scale=alt.Scale(domain=[0, 0.4])),
    y=alt.Y('RMSE_Test:Q', title='RMSE Test', scale=alt.Scale(domain=[30, 45])),
    color=alt.condition(
        model_selector,
        alt.Color('Tipo:N', legend=None),
        alt.value('lightgray')
    ),
    size=alt.condition(
        alt.datum.Es_Mejor,
        alt.value(300),
        alt.value(120)
    ),
    stroke=alt.condition(
        alt.datum.Es_Mejor,
        alt.value('gold'),
        alt.value('white')
    ),
    strokeWidth=alt.value(2),
    opacity=alt.condition(
        alt.datum.Entrenado_Actual,
        alt.value(1),
        alt.value(0.6)
    ),
    tooltip=[
        alt.Tooltip('Modelo:N'),
        alt.Tooltip('R2_Test:Q', title='R² Test', format='.4f'),
        alt.Tooltip('RMSE_Test:Q', title='RMSE Test', format='.2f'),
        alt.Tooltip('Entrenado_Actual:N', title='Entrenado Kaggle')
    ]
).properties(
    title="R² vs RMSE (Mejor = arriba-derecha)",
    width=300,
    height=200
)

# Panel 3: Overfitting Analysis
overfitting_chart = alt.Chart(modelo_metrics).mark_circle(size=120).add_selection(
    model_selector
).encode(
    x=alt.X('Overfitting_RMSE:Q', 
            title='Overfitting RMSE (Train - Test)',
            scale=alt.Scale(domain=[-30, 10])),
    y=alt.Y('Overfitting_R2:Q', 
            title='Overfitting R² (Train - Test)',
            scale=alt.Scale(domain=[0, 0.8])),
    color=alt.condition(
        model_selector,
        alt.Color('Tipo:N', legend=None),
        alt.value('lightgray')
    ),
    size=alt.condition(
        alt.datum.Es_Mejor,
        alt.value(300),
        alt.value(120)
    ),
    stroke=alt.condition(
        alt.datum.Es_Mejor,
        alt.value('gold'),
        alt.value('white')
    ),
    strokeWidth=alt.value(2),
    opacity=alt.condition(
        alt.datum.Entrenado_Actual,
        alt.value(1),
        alt.value(0.6)
    ),
    tooltip=[
        alt.Tooltip('Modelo:N'),
        alt.Tooltip('Overfitting_RMSE:Q', title='Overfitting RMSE', format='.2f'),
        alt.Tooltip('Overfitting_R2:Q', title='Overfitting R²', format='.4f')
    ]
).properties(
    title="Análisis de Overfitting (Ideal = origen)",
    width=300,
    height=200
)

# Líneas de referencia para overfitting
ref_lines = alt.Chart(pd.DataFrame({
    'x': [0, 0, -30, 0],
    'y': [0, 0.8, 0, 0],
    'type': ['vertical', 'vertical', 'horizontal', 'horizontal']
})).mark_rule(
    strokeDash=[3, 3],
    color='gray'
).encode(
    x='x:Q',
    y='y:Q'
)

# Panel 4: Distribución de Errores (basado en datos reales del test)
error_dist_data = pd.DataFrame({
    'Error_Absoluto': np.abs(y_reg_test - y_reg_pred),
    'Modelo': 'Gradient Boosting (Actual)'
})

error_histogram = alt.Chart(error_dist_data).mark_bar(
    opacity=0.8,
    binSpacing=2
).encode(
    x=alt.X('Error_Absoluto:Q', 
            bin=alt.Bin(maxbins=20, extent=[0, 100]),
            title='Error Absoluto (minutos)'),
    y=alt.Y('count():Q', title='Frecuencia'),
    color=alt.value('#2ca02c'),
    tooltip=[
        alt.Tooltip('count():Q', title='Cantidad'),
        alt.Tooltip('Error_Absoluto:Q', bin=True, title='Rango Error')
    ]
).properties(
    title="Distribución de Errores - Modelo Actual (GB)",
    width=300,
    height=200
)

# Combinar en dashboard
top_row = alt.hconcat(rmse_chart, scatter_metrics)
bottom_row = alt.hconcat(overfitting_chart + ref_lines, error_histogram)

viz3 = alt.vconcat(
    top_row,
    bottom_row
).resolve_scale(
    color='independent'
).properties(
    title=alt.TitleParams(
        text=["Dashboard Comparativo: Gradient Boosting vs Modelos Anteriores",
              "⭐ = Mejor | Opacidad = Entrenado en Kaggle actual vs histórico"],
        fontSize=16,
        anchor='start'
    )
)

print("📊 VISUALIZACIÓN 3: Dashboard Multi-Panel (CORREGIDO)")
print("   ✅ Gradient Boosting marcado como mejor modelo")
print("   ✅ Otros modelos muestran para comparación histórica")
viz3

## 5. RESUMEN DE HALLAZGOS VISUALES

### 📊 Insights de las Visualizaciones

**Visualización 1 - Scatter Plot Interactivo:**
- Los puntos cerca de la línea diagonal indican predicciones precisas
- Superficie Hard muestra mayor dispersión que Clay
- Grand Slams (partidos largos) son más difíciles de predecir

**Visualización 2 - Heatmap de Performance:**
- Grand Slams en todas las superficies muestran mayor error
- Clay tiene mejor predictibilidad que Hard/Grass
- Challengeres y Futures son más predecibles que torneos grandes

**Visualización 3 - Dashboard Comparativo:**
- Gradient Boosting ofrece el mejor balance precisión-generalización
- XGBoost sufre overfitting severo (bueno en train, malo en test)
- Ridge es estable pero con mayor error

### 🎯 Aplicación de Principios de Gramática de Gráficos

1. **Expresividad:** Cada visual comunica aspectos específicos sin ambigüedad
2. **Comparabilidad:** Escalas y referencias permiten comparación directa
3. **Interactividad:** Selecciones y filtros facilitan exploración detallada
4. **Adaptabilidad:** Diseños responsive que se adaptan a diferentes variables

In [ ]:
# Guardar datos para la aplicación Streamlit
print("💾 Guardando datos procesados para Streamlit...")

# Dataset con predicciones para Streamlit
df_streamlit = pd.DataFrame({
    'duracion_real': y_reg_test,
    'duracion_predicha': y_reg_pred,
    'categoria_real': y_class_test,
    'categoria_predicha': y_class_pred,
    'superficie': X_test['surface'].values,
    'nivel_torneo': X_test['tourney_level'].values,
    'ronda': X_test['round'].values,
    'mejor_de': X_test['best_of'].values,
    'rank_diff': X_test['rank_diff'].values,
    'is_grand_slam': X_test['is_grand_slam'].values
})

df_streamlit.to_csv('data_for_streamlit.csv', index=False)

# Métricas para mostrar en Streamlit
metrics_summary = {
    'rmse': float(rmse),
    'r2': float(r2),
    'accuracy': float(accuracy),
    'mae': float(np.mean(np.abs(y_reg_test - y_reg_pred))),
    'total_matches': len(y_reg_test)
}

import json
with open('metrics_summary.json', 'w') as f:
    json.dump(metrics_summary, f)

print(f"✅ Archivos guardados:")
print(f"   - data_for_streamlit.csv ({len(df_streamlit)} registros)")
print(f"   - metrics_summary.json (5 métricas)")
print(f"   - RMSE: {rmse:.2f} min")
print(f"   - Accuracy: {accuracy:.3f}")